In [ ]:
import pandas as pd
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression

# =============================================================================
# Custom Leave-One-Out Encoder Transformer
# =============================================================================
class LeaveOneOutEncoder(BaseEstimator, TransformerMixin):
    """
    A custom transformer that computes a leave-one-out target encoding for a given column.
    For training data, each row’s encoding is computed by excluding its own target value.
    For new (test) data, the encoding is simply the overall group average computed from training.
    """
    def __init__(self, column):
        self.column = column

    def fit(self, X, y):
        # Make a copy and attach the target
        X_temp = X.copy()
        X_temp['_target'] = y
        # Global mean of the target (used as a fallback)
        self.global_mean_ = y.mean()
        # Compute per-category sum and count of the target
        self.stats_ = X_temp.groupby(self.column)['_target'].agg(['sum', 'count'])
        return self

    def _transform_training(self, X, y):
        # For training data, compute leave-one-out values.
        X_temp = X.copy()
        X_temp['_target'] = y

        # Define a function that computes the leave-one-out encoding for a row
        def loo_encoding(row):
            group = row[self.column]
            s = self.stats_.loc[group, 'sum']
            c = self.stats_.loc[group, 'count']
            if c <= 1:
                # Not enough data to leave one out; use global mean.
                return self.global_mean_
            else:
                return (s - row['_target']) / (c - 1)

        # Create a new column with the encoded values.
        X_temp[self.column + '_loo'] = X_temp.apply(loo_encoding, axis=1)
        X_temp.drop(columns=['_target'], inplace=True)
        return X_temp

    def fit_transform(self, X, y):
        # Fit and then return training data with the new encoded column.
        self.fit(X, y)
        return self._transform_training(X, y)

    def transform(self, X):
        # For new data (e.g. test data), map each category to its average target computed during training.
        X_temp = X.copy()
        mapping = (self.stats_['sum'] / self.stats_['count']).to_dict()
        X_temp[self.column + '_loo'] = X_temp[self.column].map(mapping)
        X_temp[self.column + '_loo'].fillna(self.global_mean_, inplace=True)
        return X_temp

# =============================================================================
# A Helper Transformer to Encapsulate Feature Engineering
# =============================================================================
class FeatureEngineer(BaseEstimator, TransformerMixin):
    """
    A transformer that applies the LeaveOneOutEncoder to a specified column.
    You could expand this class to include additional feature engineering steps.
    """
    def __init__(self, column):
        self.column = column
        self.encoder = LeaveOneOutEncoder(column=self.column)

    def fit(self, X, y):
        self.encoder.fit(X, y)
        return self

    def transform(self, X):
        return self.encoder.transform(X)

# =============================================================================
# Sample Data Preparation
# =============================================================================
# Create a simple example DataFrame.
data = {
    'subject': ['Math', 'Science', 'Math', 'English', 'Science', 'Math', 'English', 'Science'],
    'feature1': [1, 2, 3, 4, 5, 6, 7, 8],
    'score': [80, 85, 78, 90, 88, 75, 92, 87]  # target variable
}
df = pd.DataFrame(data)

# Separate features and target.
X = df[['subject', 'feature1']]
y = df['score']

# Split into training and test sets.
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

# =============================================================================
# Building the Pipeline
# =============================================================================
# We want to apply our custom feature engineering to create the new column 'subject_loo'
# and then select features for our regression model (e.g., a numerical feature and the encoded subject).

pipeline = Pipeline([
    # Step 1: Apply feature engineering (i.e. leave-one-out encoding).
    ('feature_engineering', FeatureEngineer(column='subject')),

    # Step 2: Use ColumnTransformer to select the desired features.
    ('column_selector', ColumnTransformer([
         # Pass through the numerical feature
         ('num_features', 'passthrough', ['feature1']),
         # Pass through the newly engineered feature
         ('loo_feature', 'passthrough', ['subject_loo'])
    ])),

    # Step 3: Fit a linear regression model.
    ('regressor', LinearRegression())
])

# =============================================================================
# Fit the Pipeline and Predict
# =============================================================================
# For training data, the encoder uses leave-one-out encoding (each row’s target is left out).
pipeline.fit(X_train, y_train)

# For test data, the encoder maps each category to the average computed from the training set.
y_pred = pipeline.predict(X_test)
print("Predictions on test set:", y_pred)


Predictions on test set: [87.59990853 78.88764102]
